In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Create Required Project Directories
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# 2. Generate E-Commerce Dataset (1,500 online shoppers)
np.random.seed(42)
n = 1500

admin = np.random.randint(0, 20, size=n)
admin_dur = np.random.uniform(0, 1000, size=n).round(2)
info = np.random.randint(0, 10, size=n)
info_dur = np.random.uniform(0, 500, size=n).round(2)
prod = np.random.randint(1, 100, size=n)
prod_dur = np.random.uniform(10, 5000, size=n).round(2)

bounce_rates = np.random.uniform(0.0, 0.2, size=n).round(4)
exit_rates = np.random.uniform(0.0, 0.2, size=n).round(4)
page_values = np.random.uniform(0.0, 100.0, size=n).round(2)
special_day = np.random.choice([0.0, 0.2, 0.4, 0.6, 0.8, 1.0], size=n)

month = np.random.choice(['Jan', 'Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], size=n)
visitor_type = np.random.choice(['Returning_Visitor', 'New_Visitor', 'Other'], size=n, p=[0.8, 0.15, 0.05])
weekend = np.random.choice([False, True], size=n, p=[0.75, 0.25])

# Purchase intention target driven by page values & engagement
logit = 0.05 * page_values + 0.02 * prod - 10 * exit_rates - 2.0
prob = 1 / (1 + np.exp(-logit))
revenue = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'Administrative': admin,
    'Administrative_Duration': admin_dur,
    'Informational': info,
    'Informational_Duration': info_dur,
    'ProductRelated': prod,
    'ProductRelated_Duration': prod_dur,
    'BounceRates': bounce_rates,
    'ExitRates': exit_rates,
    'PageValues': page_values,
    'SpecialDay': special_day,
    'Month': month,
    'VisitorType': visitor_type,
    'Weekend': weekend,
    'Revenue': revenue
})

# Save dataset
df.to_csv('data/online_shoppers_intention.csv', index=False)
print("✅ Saved 'data/online_shoppers_intention.csv' successfully!")

# 3. Preprocessing & Pipeline Setup
X = df.drop(columns=['Revenue'])
y = df['Revenue']

num_cols = ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
            'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
cat_cols = ['Month', 'VisitorType', 'Weekend']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_tr_trans = preprocessor.fit_transform(X_train)
X_te_trans = preprocessor.transform(X_test)

# Save preprocessor artifact
joblib.dump(preprocessor, 'models/preprocessor.pkl')
print("✅ Saved 'models/preprocessor.pkl' successfully!")

# 4. Hyperparameter Tuning with RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_tr_trans, y_train)
best_rf = rf_search.best_estimator_

# Save model artifact
joblib.dump(best_rf, 'models/purchase_prediction_model.pkl')
print("✅ Saved 'models/purchase_prediction_model.pkl' successfully!")

# 5. Model Evaluation
y_pred = best_rf.predict(X_te_trans)
y_proba = best_rf.predict_proba(X_te_trans)[:, 1]

print("\n📊 Model Evaluation Performance:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# 6. Feature Importance Analysis
cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = list(cat_encoder.get_feature_names_out(cat_cols))
all_features = num_cols + cat_feature_names

importances = best_rf.feature_importances_
feature_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

print("\n🔥 Top 5 Drivers of Purchase Intention:")
print(feature_imp.head(5))

# Generate RESULTS.md
with open('reports/RESULTS.md', 'w') as f:
    f.write("# Task 9: Optimized Classification Model Results\n\n")
    f.write(f"- **Best Model:** Random Forest Classifier\n")
    f.write(f"- **ROC-AUC Score:** {roc_auc_score(y_test, y_proba):.4f}\n\n")
    f.write("## Top 5 Drivers of Customer Purchase Intention\n")
    for feat, val in feature_imp.head(5).items():
        f.write(f"1. **{feat}**: {val:.4f}\n")

print("✅ Saved 'reports/RESULTS.md' successfully!")

✅ Saved 'data/online_shoppers_intention.csv' successfully!
✅ Saved 'models/preprocessor.pkl' successfully!
✅ Saved 'models/purchase_prediction_model.pkl' successfully!

📊 Model Evaluation Performance:
              precision    recall  f1-score   support

           0       0.96      0.87      0.91       117
           1       0.92      0.98      0.95       183

    accuracy                           0.94       300
   macro avg       0.94      0.92      0.93       300
weighted avg       0.94      0.94      0.94       300

ROC-AUC Score: 0.9892

🔥 Top 5 Drivers of Purchase Intention:
PageValues                 0.606652
ExitRates                  0.085414
ProductRelated             0.082320
BounceRates                0.036474
Administrative_Duration    0.033847
dtype: float64
✅ Saved 'reports/RESULTS.md' successfully!
